# T05 CP3 | Online Retail II: EDA, Bronze, and Silver

Team 5: Boomikaa P, Chahatleen Kaur, Saba Gul, Vanshika Malviya, K. Shreya, Tushar Rathor.

This notebook starts with **exploratory data analysis (EDA)**: a first look at dataset size, shape, missing values, duplicates, date range, and unusual values. After EDA, it loads unchanged rows into Bronze and cleans them into Silver. Gold tables and the dashboard are checkpoint 4 work.

Dataset: Chen (2012), UCI Online Retail II, CC BY 4.0, https://doi.org/10.24432/C5CG6D. Export each Excel sheet as CSV and upload both files to a Unity Catalog Volume.

## 1. Set file paths and table names
Replace `<catalog>` and `<volume>`. A schema is a named group of Databricks tables.

In [ ]:
from pyspark.sql import functions as F

CATALOG = "retail_catalog"
SCHEMA = "team5_retail"
VOLUME = "retail_files"
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
FILES = [f"{BASE}/online_retail_2009_2010.csv",
         f"{BASE}/online_retail_2010_2011.csv"]
DB = f"{CATALOG}.{SCHEMA}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")


## 2. Load raw CSVs for EDA
We read fields as text first to avoid changing source values. A Spark DataFrame is a distributed table. `source_file` records data lineage: where each row came from.

In [ ]:
parts = []
for file_path in FILES:
    part = (spark.read.option("header", True).option("mode", "PERMISSIVE")
            .csv(file_path).withColumn("source_file", F.lit(file_path)))
    parts.append(part)
raw = parts[0].unionByName(parts[1], allowMissingColumns=True)
raw.printSchema()
raw.show(10, truncate=False)


## 3. EDA before any cleaning
This profile is calculated on raw rows before deduplication or type conversion. Counts and rates show what Silver must handle.

In [ ]:
profile = raw.agg(
    F.count("*").alias("rows"),
    F.countDistinct("Invoice").alias("distinct_invoices"),
    F.countDistinct("StockCode").alias("distinct_stock_codes"),
    F.countDistinct("Customer ID").alias("distinct_customers"),
    F.countDistinct("Country").alias("distinct_countries"),
    F.sum(F.col("Customer ID").isNull().cast("long")).alias("missing_customer_ids"),
    F.sum((F.col("Description").isNull() | (F.trim("Description") == "")).cast("long")).alias("blank_descriptions"),
    F.sum(F.upper(F.trim("Invoice")).startswith("C").cast("long")).alias("cancellation_rows"),
    F.sum((F.col("Quantity").cast("double") < 0).cast("long")).alias("negative_quantity_rows"),
    F.sum((F.col("Price").cast("double") <= 0).cast("long")).alias("zero_or_negative_price_rows"),
    F.min(F.to_timestamp("InvoiceDate")).alias("first_invoice_date"),
    F.max(F.to_timestamp("InvoiceDate")).alias("last_invoice_date"),
    F.min(F.col("Quantity").cast("double")).alias("minimum_quantity"),
    F.max(F.col("Quantity").cast("double")).alias("maximum_quantity"),
    F.min(F.col("Price").cast("double")).alias("minimum_price"),
    F.max(F.col("Price").cast("double")).alias("maximum_price")
).first()
for name, value in profile.asDict().items():
    print(f"{name}: {value}")
if profile["rows"]:
    print("missing customer ID rate (%):", round(100 * profile["missing_customer_ids"] / profile["rows"], 2))

# Count exact repeated source rows before cleaning.
source_columns = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"]
raw_duplicates = raw.select(*source_columns).count() - raw.select(*source_columns).dropDuplicates().count()
print("exact duplicate rows:", raw_duplicates)
raw.groupBy("Country").count().orderBy(F.desc("count")).show(10, truncate=False)
raw.select(F.col("Quantity").cast("double").alias("Quantity"),
           F.col("Price").cast("double").alias("Price")).describe().show()


## 4. Bronze: preserve raw rows
Bronze is the raw lakehouse layer. A lakehouse stores data in managed tables and supports analytics; Delta is Databricks’ reliable table format. We add a load timestamp.

In [ ]:
bronze = raw.withColumn("loaded_at", F.current_timestamp())
bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.bronze_retail_raw")
print("Bronze raw rows:", profile["rows"])


## 5. Silver: standardize and split cancellations
Silver is the cleaned layer. We trim text, convert types, remove exact duplicate rows, flag missing IDs and unusual values, and separate invoice numbers beginning with C.

In [ ]:
renames = [("Invoice", "invoice"), ("StockCode", "stock_code"),
           ("Description", "description"), ("Quantity", "quantity"),
           ("InvoiceDate", "invoice_date"), ("Price", "price"),
           ("Customer ID", "customer_id"), ("Country", "country")]
silver = raw
for old_name, new_name in renames:
    silver = silver.withColumnRenamed(old_name, new_name)
silver = (silver.withColumn("invoice", F.trim("invoice"))
    .withColumn("stock_code", F.upper(F.trim("stock_code")))
    .withColumn("description", F.trim(F.regexp_replace("description", r"\s+", " ")))
    .withColumn("country", F.initcap(F.trim("country")))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("customer_id", F.col("customer_id").cast("long"))
    .withColumn("invoice_date", F.coalesce(F.to_timestamp("invoice_date", "M/d/yyyy H:mm"), F.to_timestamp("invoice_date")))
    .dropDuplicates()
    .withColumn("is_cancelled", F.upper(F.col("invoice")).startswith("C"))
    .withColumn("missing_customer_id", F.col("customer_id").isNull())
    .withColumn("zero_or_negative_price", F.col("price").isNull() | (F.col("price") <= 0))
    .withColumn("negative_quantity", F.col("quantity") < 0))
cancellations = silver.filter(F.col("is_cancelled"))
silver_sales = silver.filter(~F.col("is_cancelled"))
cancellations.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.silver_cancellations")
silver_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.silver_retail_sales")


## 6. Save a data quality log
A data quality check measures whether values are complete and sensible. The log captures row counts, missing customer rate, and unusual price or quantity counts. These values are flagged, not silently removed.

In [ ]:
# One Silver aggregate collects the quality counts.
stats = silver.agg(
    F.count("*").alias("silver_rows_after_deduplication"),
    F.sum(F.col("is_cancelled").cast("long")).alias("cancellation_rows"),
    F.sum((~F.col("is_cancelled") & F.col("missing_customer_id")).cast("long")).alias("missing_customer_id_rows"),
    F.sum((~F.col("is_cancelled") & F.col("zero_or_negative_price")).cast("long")).alias("zero_or_negative_price_rows"),
    F.sum((~F.col("is_cancelled") & F.col("negative_quantity")).cast("long")).alias("negative_quantity_rows")
).first()
sales_rows = (stats["silver_rows_after_deduplication"] or 0) - (stats["cancellation_rows"] or 0)
missing_rows = stats["missing_customer_id_rows"] or 0
quality_rows = [
    ("raw_input_rows", str(profile["rows"])),
    ("exact_duplicate_rows_removed", str(raw_duplicates)),
    ("silver_sales_output_rows", str(sales_rows)),
    ("cancellation_rows", str(stats["cancellation_rows"] or 0)),
    ("missing_customer_id_rows", str(missing_rows)),
    ("missing_customer_id_rate_percent", str(round(100 * missing_rows / sales_rows, 2) if sales_rows else 0)),
    ("zero_or_negative_price_rows", str(stats["zero_or_negative_price_rows"] or 0)),
    ("negative_quantity_rows", str(stats["negative_quantity_rows"] or 0)),
]
quality_log = spark.createDataFrame(quality_rows, ["check_name", "check_value"])
quality_log = quality_log.withColumn("logged_at", F.current_timestamp())
quality_log.write.format("delta").mode("overwrite").saveAsTable(f"{DB}.silver_quality_log")
quality_log.show(truncate=False)


## EDA findings from the provided workbook

A full local scan found 1,067,371 rows across 1 Dec 2009–9 Dec 2011; 243,007 missing customer IDs (22.77%); 4,382 blank descriptions; 19,494 cancellation rows; 34,335 exact repeated rows; 22,950 negative quantities; and 6,207 rows with zero or negative price. The United Kingdom has the most records (981,330, about 91.94%). These are data quality observations, not business recommendations. The notebook recalculates the profile from uploaded CSVs; review those results because CSV export settings can affect date/value parsing.

Run a small sample first, then run the full dataset once to respect the Databricks Free Edition quota. This notebook needs Databricks, PySpark, and a Unity Catalog Volume; it is not plain local Python. Checkpoint 3 ends at Silver; Gold tables/dashboard are checkpoint 4.